# Part 14 - The Supervisor and the Handoff: multi-agent, and when it hurts

*Agents from First Principles, Part 14.*

**The frontier track continues.** Every part so far had ONE agent with ONE context window. That is a real limit. A breadth-first task with independent parts cannot be parallelized inside one context, and a task that should really go to a specialist is awkward to cram into a generalist's loop. The fix is to use MORE THAN ONE agent. The fix is also a trap: multi-agent systems are more expensive, harder to debug, and frequently OVERKILL. This part builds the two main multi-agent shapes and, just as importantly, the honest economics of when NOT to reach for them.

1. **Orchestrator-worker (the supervisor).** A lead agent DECOMPOSES a task into subtasks, spawns N WORKERS, each with its OWN isolated context and a TYPED BRIEF (objective, output format, tool guidance, boundaries), and then SYNTHESIZES their results. The worker is just the Part 1 agent reused as a black box; we do not rebuild the loop. Workers coordinate through a shared append-only BLACKBOARD.

2. **Handoff as a tool call.** Sometimes the right move is not to fan out but to TRANSFER CONTROL to a specialist, carrying the live trace along. `handoff(to=...)` is a tool like any other. We show a clean handoff and its two classic failure modes: TRUNCATION (the trace is cut, so the specialist is missing the key fact) and ROLE CONFUSION (the brief does not say who owns the task, so it stalls).

3. **The economics: single vs supervisor vs debate**, on one task, with the numbers. A proposer-critic DEBATE shows the uncomfortable truth that more rounds is NOT more quality: round 3 REGRESSES. The single agent is cheapest to wire; the supervisor isolates context and parallelizes; debate is the most expensive and the least monotonic.

> **HONESTY about parallelism (the feasibility crux).** The offline default runs the workers SEQUENTIALLY. We never print a fabricated wall-clock speedup. What multi-agent buys is reported honestly as (a) LLM-CALL COUNT and (b) TOKEN ISOLATION: each worker sees only its own small brief, not one ever-growing shared transcript. With real concurrency the independent workers run in a single round (depth 1, the Part 3 idea). A runaway supervisor is stopped by Part 8's circuit breaker (reused, not rebuilt), and these distributed runs render as the Part 11 span tree.

> **Runs with no network, no API key, and no third-party dependency.** The deterministic lead/workers/critic are the offline source of truth; `generate()` (a real LLM as the lead/worker/critic) is one flag away. Set `OPENAI_API_KEY` to see the real banner; the logic always falls through to the deterministic path so output stays reproducible.

Companion script: `supervisor_and_handoffs.py`

## Setup

One standard-library import does the work: `os` lets us check for an API key without ever requiring one. No third-party package is needed; every cell runs fully offline, exactly as in Parts 1 to 13. The world is the support world carried from RAG and Parts 1 to 13: refunds, the earbuds warranty, shipping.

In [ ]:
import os

print("ready -- no network, no API key, no third-party dependency required")

## What is reused, and what is new

This part is an orchestration layer ABOVE the single agent, so almost nothing is rebuilt:

- **The worker IS the Part 1 agent**, reused as a black box. Hand it a typed brief, it runs its one tool and returns one result. We never re-derive the reason/act/observe loop here; that was Part 1.
- **The circuit breaker is Part 8, reused not rebuilt.** A runaway supervisor (workers that spawn workers, unbounded fan-out) is stopped by the same `BudgetMeter` idea from Part 8; we point at it rather than duplicating it.
- **These distributed runs render as the Part 11 span tree** (a forward reference to that observability part).

What is genuinely NEW, and absent from the whole RAG series (which was single-agent end to end): the orchestration itself, parallel workers with SEPARATE contexts and typed briefs, agent-control HANDOFF (categorically different from RAG's complexity router and text-vs-SQL router, which route a query inside one agent), the shared BLACKBOARD, and the single-vs-multi ECONOMICS.

## Step 0 - The tools (the worker's action space)

The worker's action space is two retrieval tools over the support world, the same shape the series has used throughout (and which would travel over MCP as in Part 12). The worker is the Part 1 agent reused as a black box: hand it a brief, it returns one result. `search_policy` answers refund and shipping questions; `search_products` answers the earbuds warranty. `TOOLS` is the dispatch table the worker indexes by its brief's `tool` field.

In [ ]:
def search_policy(query):
    if "shipping" in query:
        return "Standard shipping is 3 to 5 business days; express is next business day."
    return "Refunds within 30 days; a 10% restocking fee applies after the window."


def search_products(query):
    return "The Globex earbuds carry a 2-year limited warranty."


TOOLS = {"search_policy": search_policy, "search_products": search_products}

print("tools defined:", ", ".join(TOOLS))

## Step 1 - The blackboard and the worker (isolated context)

The orchestrator-worker shape has three moving parts: the supervisor DECOMPOSES into typed briefs, each WORKER runs in its OWN context (it only ever sees its brief), writing to a shared BLACKBOARD, and the supervisor SYNTHESIZES the blackboard into one result.

`Blackboard` is an append-only shared workspace: each worker `post`s its `(author, text)` result so the supervisor can read them all back in order. The `worker` is the Part 1 agent reused as a black box: its context is JUST this brief (that is the isolation), it runs its one tool, posts the result, and reports how many tokens its tiny context held. The token count, `len(objective) + len(query) + 8`, is the honest "token isolation" number: each worker's context is small and constant, NOT a transcript that grows with every step.

In [ ]:
class Blackboard:
    def __init__(self):
        self.entries = []

    def post(self, author, text):
        self.entries.append((author, text))


def worker(brief, blackboard):
    """A black-box Part 1 agent. Its context is JUST this brief (isolation). It runs
    its one tool and posts to the blackboard."""
    result = TOOLS[brief["tool"]](brief["query"])
    blackboard.post(brief["id"], result)
    ctx_tokens = len(brief["objective"].split()) + len(brief["query"].split()) + 8
    return result, ctx_tokens


print("Blackboard and worker ready (each worker context is isolated and small).")

## Step 2 - The supervisor: decompose, fan out, synthesize

`supervisor_run` is the lead agent. It DECOMPOSES the goal into three TYPED BRIEFS, each a dict with an `objective`, an `output` format, a `tool` and `query` (tool guidance), and `boundaries`. That typed brief is exactly the contract that keeps a worker on-task: it knows what to produce, in what shape, with which tool, and what is out of scope.

It then fans out: each brief runs through a `worker` in its own isolated context, posting to the blackboard, and we sum the per-worker context tokens. Finally the supervisor SYNTHESIZES, joining the blackboard entries into one briefing. Note the honesty baked in: the loop is SEQUENTIAL here, but the three briefs are INDEPENDENT, so with real concurrency they would run in ONE round (depth 1, the Part 3 idea). We report the call count and token isolation, never a wall-clock speedup.

In [ ]:
def supervisor_run(goal):
    print(f"  supervisor decomposes: {goal!r}")
    briefs = [
        {"id": "w1", "objective": "summarize the refund policy", "output": "one sentence",
         "tool": "search_policy", "query": "refund policy", "boundaries": "policy only"},
        {"id": "w2", "objective": "state the earbuds warranty", "output": "one sentence",
         "tool": "search_products", "query": "earbuds warranty", "boundaries": "products only"},
        {"id": "w3", "objective": "list the shipping options", "output": "one sentence",
         "tool": "search_policy", "query": "shipping options", "boundaries": "shipping only"},
    ]
    bb = Blackboard()
    total_ctx = 0
    for b in briefs:
        result, ctx = worker(b, bb)
        total_ctx += ctx
        print(f"    [{b['id']}] brief: {b['objective']} (own context, {ctx} tokens) -> {result}")
    briefing = " ".join(text for _author, text in bb.entries)
    print(f"  supervisor synthesizes the blackboard into the briefing:")
    print(f"    {briefing}")
    return briefing, total_ctx


print("supervisor_run ready (decompose -> isolated workers -> blackboard -> synthesize).")

## Step 3 - Handoff as a tool call (and its two failure modes)

Sometimes the right move is not to fan out but to TRANSFER CONTROL to a specialist, carrying the live trace along. `handoff(to=...)` is a tool like any other: it passes the running trace to a sub-agent that takes over.

A CLEAN handoff carries the full trace and a clear role, and the specialist can act. The two classic failure modes are both visible here:

- **TRUNCATION** (`truncate=True`): the trace is cut to its last entry, but the key fact (the order id) was logged EARLIER. The specialist receives only the tail, cannot find the order id, and fails. This is the silent killer of handoffs: the carried context is incomplete.
- **ROLE CONFUSION** (`role_clear=False`): the full trace is carried, but the brief never said who OWNS the task. The specialist and the lead both wait for the other, and it stalls. Control transferred, but ownership did not.

The function prints what it carries (`N/M trace entries`, `role=clear|UNCLEAR`) so you can see the cause of each outcome.

In [ ]:
def handoff(to, trace, role, truncate=False, role_clear=True):
    carried = trace[-1:] if truncate else trace
    print(f"    handoff(to={to!r}): carrying {len(carried)}/{len(trace)} trace entries, "
          f"role={'clear' if role_clear else 'UNCLEAR'}")
    if not role_clear:
        return f"[stalled] {to} and the lead both wait: the brief never said who owns the task."
    order = next((t.split("order ")[1].split()[0] for t in carried if "order " in t), None)
    if order is None:
        return f"[failed] {to} cannot act: the order id was truncated out of the carried trace."
    return f"[done] {to} refunded order {order} using the carried trace."


print("handoff ready (clean + truncation + role-confusion paths).")

## Step 4 - The economics: single vs supervisor vs debate

Three strategies for the SAME task, with the numbers. We report LLM CALLS and total context TOKENS (the token-isolation story) and a `quality` out of 3, never a wall-clock speedup.

- **`single_agent`**: one agent, one GROWING context. Each of the four steps (refund, warranty, shipping, synthesize) re-sends everything before it, so the per-step context climbs and the total tokens add up. Fewest calls, but the context grows the whole way (Parts 3 and 11).
- **`supervisor_econ`**: decompose + 3 isolated workers + synthesize = 5 calls. More calls than single, but each worker context is small AND CONSTANT (it does not grow), so the total tokens are lower, and the independent workers parallelize.
- **`debate_econ`**: a proposer then three critic-revise rounds. The key, uncomfortable result: quality is NOT monotonic in rounds. The proposer is 2/3, rounds 1 and 2 reach 3/3, and round 3 REGRESSES back to 2/3. Debate is the most expensive (most calls, fastest-growing transcripts) and the least reliable.

In [ ]:
def single_agent(goal):
    """One agent, one GROWING context: each step re-sends everything before it."""
    steps = ["refund policy", "earbuds warranty", "shipping options", "synthesize"]
    ctx, total = 0, 0
    for i, _s in enumerate(steps, start=1):
        ctx += 30                            # the transcript grows every step (Parts 3/11)
        total += ctx
    return {"calls": len(steps), "tokens": total, "quality": 3}


def supervisor_econ(goal):
    # decompose + 3 isolated workers + synthesize; each worker context is small + constant
    calls = 1 + 3 + 1
    tokens = 30 + (40 + 40 + 40) + 70        # isolated worker contexts do not grow
    return {"calls": calls, "tokens": tokens, "quality": 3}


def debate_econ(goal):
    """Proposer then 3 critic-revise rounds. Quality is NOT monotonic in rounds."""
    rounds = [("proposer", 2), ("round 1", 3), ("round 2", 3), ("round 3", 2)]  # round 3 regresses
    calls = 1 + 3 * 2                        # proposer + 3x(critic+revise)
    tokens = 0
    base = 50
    for i, (_name, _q) in enumerate(rounds):
        tokens += base + i * 20              # debate transcripts grow fast
    return {"calls": calls, "tokens": tokens, "quality_by_round": rounds,
            "quality": rounds[-1][1]}


print("single_agent, supervisor_econ, debate_econ ready.")

## Step 5 - generate(): the real LLM as lead/worker/critic (reference shape only)

Same device as Parts 1 to 13. On the real path the lead, the workers, and the critic are all LLMs: the supervisor's decomposition and synthesis, each worker's tool-using loop, and the debate's proposer/critic are real model calls. `generate()` is the single swappable call that lights up that path; the offline demo never calls it, so output stays reproducible. SDK names and model ids move fast; check current docs. Only `generate()` would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: a hosted LLM is the lead/worker/critic. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[orchestrator] OPENAI_API_KEY set; a real LLM would be the lead/workers/critic. "
          "Falling through to deterministic logic for reproducibility.")
else:
    print("[orchestrator] no OPENAI_API_KEY; deterministic lead/workers (offline default)")

## Demo - the banner and the honesty up front

Everything below runs fully offline. We open with the banner and, before any result, the honesty point that governs how to read every number in this notebook: parallelism is SEQUENTIAL in this offline default, so we report LLM-call count and TOKEN ISOLATION (each worker sees only its brief), never a fabricated wall-clock speedup.

In [ ]:
bar = "=" * 72
print(bar)
print("THE SUPERVISOR AND THE HANDOFF  -  multi-agent, and when it hurts")
print(bar)
if os.environ.get("OPENAI_API_KEY"):
    print("[orchestrator] OPENAI_API_KEY set; a real LLM would be the lead/workers/critic. "
          "Falling through to deterministic logic for reproducibility.")
else:
    print("[orchestrator] no OPENAI_API_KEY; deterministic lead/workers (offline default)")
print("\nParallelism is SEQUENTIAL in this offline default: we report LLM-call count and")
print("TOKEN ISOLATION (each worker sees only its brief), never a fabricated wall-clock.")

## Demo 1 - Orchestrator-worker: decompose, fan out, synthesize

The supervisor takes one goal, "Prepare a customer briefing on refunds, the earbuds warranty, and shipping," and decomposes it into three typed briefs. Each runs in its OWN isolated context (14 tokens each) and posts its one sentence to the blackboard: w1 the refund policy, w2 the warranty, w3 shipping. The supervisor then synthesizes the blackboard into the final briefing. The closing line states the honesty point in place: three workers, each its own context, and with real concurrency they run in ONE round.

In [ ]:
print("\n" + "-" * 72)
print("1) ORCHESTRATOR-WORKER: decompose, fan out to isolated workers, synthesize.")
print("-" * 72)
goal = "Prepare a customer briefing on refunds, the earbuds warranty, and shipping."
supervisor_run(goal)
print("    (3 workers, each its own context; with real concurrency they run in ONE round.)")

## Demo 2 - Handoff as a tool call: the clean case and its two failure modes

The same trace, three handoffs. The CLEAN handoff carries all 3/3 entries with a clear role, so the billing specialist finds order `ORD-3300` and refunds it. TRUNCATION carries only 1/3 entries (the order id was logged earlier, now cut), so the specialist cannot act. ROLE CONFUSION carries the full 3/3 trace but with an UNCLEAR role, so it stalls: control moved, ownership did not. Watch the printed `N/M trace entries` and `role=` to see the cause of each outcome.

In [ ]:
print("\n" + "-" * 72)
print("2) HANDOFF AS A TOOL CALL: transfer control + the trace to a specialist.")
print("-" * 72)
trace = ["identified order ORD-3300 for refund", "looked up policy: eligible",
         "user confirmed they want the refund"]
print("  clean handoff (full trace, clear role):")
print("    -> " + handoff("billing-specialist", trace, "refund the order"))
print("  failure mode TRUNCATION (trace cut to the last entry; the order id was earlier):")
print("    -> " + handoff("billing-specialist", trace, "refund the order", truncate=True))
print("  failure mode ROLE CONFUSION (brief never said who owns it):")
print("    -> " + handoff("billing-specialist", trace, "refund the order", role_clear=False))

## Demo 3 - The economics table: single vs supervisor vs debate

The three strategies side by side on the same task. Single is the fewest calls (4) but the most tokens (300) because of its one growing context. Supervisor is more calls (5) but fewer tokens (220) because each worker context is isolated and small, and the same 3/3 quality. Debate is the most expensive (7 calls, 320 tokens) AND the lowest quality (2/3). Cost-per-success follows tokens: pay for breadth only when the task IS broad.

In [ ]:
print("\n" + bar)
print("3) THE ECONOMICS: single vs supervisor vs debate (call count + token isolation).")
print(bar)
s, sup, deb = single_agent(goal), supervisor_econ(goal), debate_econ(goal)
print(f"  {'strategy':<14}{'LLM calls':>11}{'tokens':>9}{'quality':>9}")
print("  " + "-" * 41)
print(f"  {'single':<14}{s['calls']:>11}{s['tokens']:>9}{str(s['quality']) + '/3':>9}")
print(f"  {'supervisor':<14}{sup['calls']:>11}{sup['tokens']:>9}{str(sup['quality']) + '/3':>9}")
print(f"  {'debate':<14}{deb['calls']:>11}{deb['tokens']:>9}{str(deb['quality']) + '/3':>9}")

## Demo 4 - Debate by round: more rounds is NOT more quality

The most important single result in the part. The debate's quality per round is non-monotonic: the proposer is 2/3, rounds 1 and 2 climb to 3/3, and round 3 REGRESSES back to 2/3. More rounds did not buy more quality; the third round actively made it worse. This is why debate is dear and unreliable, and why you should not keep adding critic rounds expecting steady improvement.

In [ ]:
print("\n  Debate quality by round (more rounds is NOT more quality):")
for name, q in deb["quality_by_round"]:
    flag = "  <- regressed" if name == "round 3" else ""
    print(f"    {name:<10} quality {q}/3{flag}")
print("\n  Reading it:")
print("  - single: fewest calls, but ONE growing context (tokens climb every step)")
print("  - supervisor: more calls, but each worker context is ISOLATED and small;")
print("    independent workers parallelize (one round with real concurrency)")
print("  - debate: most expensive AND non-monotonic; round 3 made it worse")
print("  - cost-per-success follows tokens: pay for breadth only when the task IS broad;")
print("    a single agent is the right, cheaper tool for a small sequential task.")

## Demo 5 - Safety: the runaway supervisor is stopped by Part 8's breaker

The closing safety note, stated as reuse not rebuild. A runaway supervisor (workers spawning workers, unbounded fan-out) is exactly the kind of loop Part 8's circuit breaker already stops; we reuse it rather than build a new one. And these distributed runs render as the Part 11 span tree, the observability view forward-referenced earlier.

In [ ]:
print("\n" + bar)
print("4) SAFETY: a runaway supervisor (workers spawning workers) is stopped by Part 8's")
print("circuit breaker, reused not rebuilt; and these runs render as the Part 11 span tree.")
print(bar)

print("\n" + bar)
print("Done. Multi-agent is a tool, not a default:")
print("  - ORCHESTRATOR-WORKER: typed briefs + isolated contexts + a blackboard + synthesis")
print("  - HANDOFF as a tool call: transfer control + trace (watch truncation + role confusion)")
print("  - the economics: supervisor isolates + parallelizes; debate is dear and not monotonic;")
print("    a single agent is best for a small task. Reach for multi-agent when the work is broad.")
print(bar)

## Wrap-up: multi-agent is a tool, not a default

A single agent has one context window; this part promoted it to a multi-agent system, and was honest about the price:

- **Orchestrator-worker (the supervisor).** A lead decomposes a task into TYPED BRIEFS, fans out to WORKERS with ISOLATED contexts that coordinate through a BLACKBOARD, then SYNTHESIZES. The worker is the Part 1 agent reused as a black box, not rebuilt.
- **Handoff as a tool call.** Transfer control AND the live trace to a specialist. Two failure modes to watch: TRUNCATION (the key fact cut out of the carried trace) and ROLE CONFUSION (the brief never said who owns the task).
- **The economics are not flattering to multi-agent.** Single is the cheapest to wire and best for a small sequential task. The supervisor costs more calls but ISOLATES context and parallelizes (one round with real concurrency). Debate is the most expensive AND non-monotonic: round 3 regressed.
- **Honesty about parallelism.** Offline the workers run SEQUENTIALLY; we report LLM-call count and TOKEN ISOLATION, never a fabricated wall-clock. With real concurrency the independent workers run in one round (depth 1, Part 3). A runaway supervisor is stopped by Part 8's breaker, reused not rebuilt, and these runs render as the Part 11 span tree.

**Part 15 - agent-to-agent (A2A).** The supervisor here owned its workers in one process. Part 15 opens the boundary: independent agents, possibly built by different teams on different stacks, that discover and call each other over a shared protocol, the agent-to-agent layer above the single-owner orchestration we built here.